<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day05-lab.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 5 lab: real database searches, in code {.unnumbered}

**Recommended: run this on the shared course server over SSH, not just
in Colab.** Every tool below (`clustalw`, `psiblast`, `apt-get`) is a
real command-line tool, and this notebook runs the same way
whether opened via VSCode's Jupyter support on the server (see the Day
2 book page's "Connecting to the shared course server" section to set
that up) or in Colab. On the server, you also get a real terminal
alongside it if you'd rather type these same commands by hand.

Three real, runnable pieces, each producing output you'll report on in
the Day 5 lab quiz on Canvas -- everything here runs against real tools
and real data, not a toy simulation:

1. **ClustalW + a real PSSM** on five real globin-family proteins already
   used throughout this course.
2. **A real PSI-BLAST search**, comparing its first iteration against its
   third, against a real (if small) local copy of every sequence in the
   PDB.
3. **A live Pfam domain search** for HIV-1 Pol via EBI's real InterPro
   API.

Run every cell top to bottom. Nothing here is a fill-in-the-blank
exercise -- just run it, read the real output, and use it to answer the
quiz.


## 1. Install the real command-line tools

Colab's stock image doesn't ship BLAST+ or ClustalW -- both are real,
standard bioinformatics packages, installed here the same way you would
on any Linux machine.


In [ ]:
import shutil, subprocess

for tool, package in [("blastp", "ncbi-blast+"), ("clustalw", "clustalw")]:
    if shutil.which(tool) is None:
        print(f"Installing {package} (provides {tool})...")
        subprocess.run(["apt-get", "install", "-y", "-qq", package], check=True)
    else:
        print(f"{tool} already available, skipping install.")


## 2. ClustalW: a real multiple sequence alignment, and a real PSSM from it

Five real human globin-family proteins (HBB, HBA1, MB, NGB, CYGB --
the same set this course has used since Day 2/Day 4) aligned with
**ClustalW**, the real command-line tool the Day 4 book page's own
"Multiple sequence alignment" section names as the classic **progressive**
aligner. Once you have a real alignment, building a PSSM from it is just
counting: for each column, how often does each amino acid appear?


In [ ]:
import subprocess
from Bio import AlignIO
from collections import Counter

GLOBIN_SEQS = {
    "HBB":  "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH",
    "HBA1": "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR",
    "MB":   "MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPETLEKFDKFKHLKSEDEMKASEDLKKHGATVLTALGGILKKKGHHEAEIKPLAQSHATKHKIPVKYLEFISECIIQVLQSKHPGDFGADAQGAMNKALELFRKDMASNYKELGFQG",
    "NGB":  "MERPEPELIRQSWRAVSRSPLEHGTVLFARLFALEPDLLPLFQYNCRQFSSPEDCLSSPEFLDHIRKVMLVIDAAVTNVEDLSSLEEYLASLGRKHRAVGVKLSSFSTVGESLLYMLEKCLGPAFTPATRAAWSQLYGAVVQAMSRGWDGE",
    "CYGB": "MEKVPGEMEIERRERSEELSEAERKAVQAMWARLYANCEDVGVAILVRFFVNFPSAKQYFSQFKHMEDPLEMERSPQLRKHACRVMGALNTVVENLHDPDKVSSVLALVGKAHALKHKVEPVYFKILSGVILEVVAEEFASDFPPETQRAWAKLRGLIYSHVTAAYKEVGWVQQVPNATTPPATLPSSGP",
}

with open("globins.fasta", "w") as f:
    for name, seq in GLOBIN_SEQS.items():
        f.write(f">{name}\n{seq}\n")

subprocess.run(["clustalw", "-INFILE=globins.fasta", "-ALIGN", "-OUTFILE=globins.aln"], check=True, capture_output=True)

aln = AlignIO.read("globins.aln", "clustal")
print(f"Real ClustalW alignment: {len(aln)} sequences, {aln.get_alignment_length()} columns")
for rec in aln:
    print(f"  {rec.id:6s} {str(rec.seq)[:60]}...")

# A PSSM, built the honest way: count how often each amino acid appears
# in each real alignment column (gaps don't count as a residue).
total_counts = Counter()
for col in range(aln.get_alignment_length()):
    column = aln[:, col]
    total_counts.update(r for r in column if r != "-")

print("\nTotal count per amino acid, summed across all real columns:")
for aa, n in total_counts.most_common(10):
    print(f"  {aa}: {n}")


## 3. A real PSI-BLAST search: iteration 1 vs. iteration 3

PSI-BLAST's whole point is that later iterations use a **profile**
(built from the previous round's hits) instead of a single fixed
substitution matrix -- the "From one sequence to many" section
of the Day 5 book page. To see this matter for real, this searches
alpha-globin (HBA1/HBA2 -- the same protein, UniProt `P69905`) against
a real local copy of **every sequence in the PDB** (`pdbaa`, ~72,000
real sequences, downloaded directly from NCBI's own public FTP site --
small enough to fetch and build fresh every time, so this is always a
real, current PDB snapshot, not a stale one).


In [ ]:
import subprocess, os, re
import requests

r = requests.get("https://rest.uniprot.org/uniprotkb/P69905.fasta", timeout=15)
r.raise_for_status()
with open("hba.fasta", "w") as f:
    f.write(r.text)
print("Query:")
print(r.text)

if not os.path.exists("pdbaa.tar.gz"):
    print("Downloading pdbaa (real, current PDB sequences, NCBI's own public FTP)...")
    r = requests.get("https://ftp.ncbi.nlm.nih.gov/blast/db/pdbaa.tar.gz", timeout=120)
    r.raise_for_status()
    with open("pdbaa.tar.gz", "wb") as f:
        f.write(r.content)
    subprocess.run(["tar", "xzf", "pdbaa.tar.gz"], check=True)

subprocess.run(
    ["psiblast", "-query", "hba.fasta", "-db", "pdbaa", "-num_iterations", "3", "-out", "psiblast_full.txt"],
    check=True, capture_output=True,
)

text = open("psiblast_full.txt").read()
rounds = re.split(r"\nResults from round (\d)\n", text)[1:]
round_texts = {rounds[i]: rounds[i + 1] for i in range(0, len(rounds), 2)}

# Modern pdbaa headers look like "1BZ1_A Chain A, ..." (PDB ID + chain),
# not the old "gi|N|pdb|ID|CHAIN" format -- verified against a real,
# freshly-downloaded pdbaa while building this notebook.
for round_num in ["1", "2", "3"]:
    section = round_texts[round_num]
    hits = re.findall(r"^(\w{4})_(\w+)\s.*?\s+([\d.]+)\s+([\d.e+-]+)\s*$", section, re.MULTILINE)
    print(f"Round {round_num}: {len(hits)} significant PDB hits")
    if hits:
        print(f"  top hit: PDB {hits[0][0]} chain {hits[0][1]}, bit score {hits[0][2]}, E-value {hits[0][3]}")


## 4. A live Pfam domain search: HIV-1 Pol

The same real, live EBI InterPro API the Day 5 book page's own worked
example uses for *HBB* -- here run on HIV-1's Gag-Pol polyprotein
(UniProt `P04585`), which (unlike a single-domain protein like HBB)
carries several real Pfam domains strung together: protease, reverse
transcriptase, RNase H, and integrase, each a real, separately-evolved
enzymatic module.


In [ ]:
import requests

r = requests.get("https://www.ebi.ac.uk/interpro/api/entry/pfam/protein/uniprot/P04585/", timeout=15)
r.raise_for_status()
results = r.json()["results"]

print(f"{len(results)} real Pfam domains found for P04585 (HIV-1 Gag-Pol polyprotein):")
print()
hits = []
for entry in results:
    meta = entry["metadata"]
    loc = entry["proteins"][0]["entry_protein_locations"][0]
    frag = loc["fragments"][0]
    hits.append((meta["accession"], meta["name"], frag["start"], frag["end"], loc["score"]))
    print(f"  {meta['accession']}  {meta['name']:38s} residues {frag['start']:>5}-{frag['end']:<5}  score={loc['score']}")

best = min(hits, key=lambda h: h[4])
print(f"\nStrongest (lowest-score) match: {best[0]} \"{best[1]}\"")


## Done

Once all three sections above have run and printed real output, answer
the Day 5 lab quiz on Canvas using your own results -- your exact
numbers may differ slightly from anyone else's (PDB and Pfam both grow
over time), which is expected and fine.
